In [ ]:
import numpy as np
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import yfinance as yf

# Initialize the ticker
ticker = yf.Ticker("MSFT")

# Get historical data (OHLCV)
# 'period' can be 1d, 5d, 1mo, 1y, 5y, max
df = ticker.history(period="20y")
df = df.reset_index()
# For forecasting, usually 'Close' or 'Adj Close' is the target
print(df.head())

                       Date       Open       High        Low      Close  \
0 2006-04-10 00:00:00-04:00  19.007953  19.154545  18.987012  19.049837   
1 2006-04-11 00:00:00-04:00  19.049842  19.070783  18.847406  18.938152   
2 2006-04-12 00:00:00-04:00  18.917202  18.987007  18.826454  18.987007   
3 2006-04-13 00:00:00-04:00  18.903242  18.987009  18.847398  18.896261   
4 2006-04-17 00:00:00-04:00  18.868344  18.882304  18.658928  18.735714   

     Volume  Dividends  Stock Splits  
0  39432000        0.0           0.0  
1  42953400        0.0           0.0  
2  32183000        0.0           0.0  
3  28160000        0.0           0.0  
4  35796200        0.0           0.0  


In [ ]:
df.isna().sum()

,0
Date,0
Open,0
High,0
Low,0
Close,0
Volume,0
Dividends,0
Stock Splits,0


In [ ]:
# Remove null values
df.dropna(inplace = True)

In [ ]:
# Viewing the statistics for the dataset
df.describe()

,Open,High,Low,Close,Volume,Dividends,Stock Splits
count,5029.000000,5029.000000,5029.000000,5029.000000,5.029000e+03,5029.000000,5029.0
mean,128.374060,129.570421,127.109464,128.388521,4.170892e+07,0.006148,0.0
std,143.254460,144.483051,141.893545,143.231820,2.775774e+07,0.056840,0.0
min,11.149860,11.457949,10.907791,11.113183,5.855900e+06,0.000000,0.0
25%,21.693169,21.894433,21.502878,21.679571,2.348570e+07,0.000000,0.0
50%,46.854094,47.197871,46.464107,46.989609,3.378170e+07,0.000000,0.0
75%,227.415644,229.901145,224.009188,227.358063,5.197770e+07,0.000000,0.0
max,552.023241,552.242002,538.530591,539.825195,5.910522e+08,0.910000,0.0


In [ ]:
df = df.reset_index()
df.head()

,index,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,0,2006-04-10 00:00:00-04:00,19.007953,19.154545,18.987012,19.049837,39432000,0.0,0.0
1,1,2006-04-11 00:00:00-04:00,19.049842,19.070783,18.847406,18.938152,42953400,0.0,0.0
2,2,2006-04-12 00:00:00-04:00,18.917202,18.987007,18.826454,18.987007,32183000,0.0,0.0
3,3,2006-04-13 00:00:00-04:00,18.903242,18.987009,18.847398,18.896261,28160000,0.0,0.0
4,4,2006-04-17 00:00:00-04:00,18.868344,18.882304,18.658928,18.735714,35796200,0.0,0.0


In [ ]:
df['return'] = df['Close'].pct_change()

In [ ]:
import plotly.graph_objects as go

fig = go.Figure(data=[go.Candlestick(
    x=df['Date'],
    open=df['Open'],
    high=df['High'],
    low=df['Low'],
    close=df['Close']
)])

fig.update_layout(
    title="Candlestick Chart - Zoomed",
    template="plotly_dark",
    # This is the important part:
    xaxis_rangeslider_visible=False
)

# Optional: Force the y-axis to stay close to the data
fig.update_yaxes(autorange=True, fixedrange=False)

fig.show()

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Calculate returns (if not already done)
df['return'] = df['Close'].pct_change()

# Create subplots: 1 row for the line, 1 for the distribution
fig = make_subplots(rows=2, cols=1,
                    subplot_titles=("Daily Returns (%)", "Distribution of Returns"),
                    vertical_spacing=0.15)

# Daily Returns Line Chart
fig.add_trace(
    go.Scatter(x=df['Date'], y=df['return'], name="Daily Return",
               line=dict(color='#00C805', width=1)),
    row=1, col=1
)

# Histogram of Returns
fig.add_trace(
    go.Histogram(x=df['return'], name="Frequency",
                 marker_color='#30B0EA', opacity=0.7),
    row=2, col=1
)

fig.update_layout(
    height=800,
    template="plotly_dark",
    showlegend=False,
    title_text="Stock Returns Analysis" ,
)

# Format y-axis as percentage for clarity
fig.update_yaxes(tickformat=".1%", row=1, col=1)

fig.show()

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Bar(
    x=df['Date'],
    y=df['Volume'],
    name='Volume',
    marker_color='#30B0EA'
))

fig.update_layout(
    title="Trading Volume Over Time",
    xaxis_title="Date",
    yaxis_title="Volume",
    template="plotly_dark"
)

fig.show()

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Create subplots: 2 rows, 1 column
# row_width=[0.2, 0.7] gives more space to the price chart (70%) than volume (20%)
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    vertical_spacing=0.03,
                    subplot_titles=('Price', 'Volume'),
                    row_width=[0.2, 0.7])

# 1. Add Candlestick to Row 1
fig.add_trace(go.Candlestick(
    x=df['Date'],
    open=df['Open'], high=df['High'],
    low=df['Low'], close=df['Close'],
    name='Price'
), row=1, col=1)

# 2. Add Volume Bars to Row 2
# We color volume bars based on whether the day was green or red
colors = ['green' if row['Close'] >= row['Open'] else 'red' for _, row in df.iterrows()]

fig.add_trace(go.Bar(
    x=df['Date'],
    y=df['Volume'],
    name='Volume',
    marker_color=colors
), row=2, col=1)

# Cleanup Layout
fig.update_layout(
    title="Price and Volume Analysis",
    template="plotly_dark",
    xaxis_rangeslider_visible=False, # Hiding slider to avoid the "squashed" look
    showlegend=False,
    height=800
)

fig.show()

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Calculate Bollinger Bands
df['SMA20'] = df['Close'].rolling(window=20).mean()
df['STD20'] = df['Close'].rolling(window=20).std()
df['Upper_Band'] = df['SMA20'] + (df['STD20'] * 2)
df['Lower_Band'] = df['SMA20'] - (df['STD20'] * 2)

# 2. Calculate Annualized Rolling Volatility
df['Returns'] = df['Close'].pct_change()
df['Volatility'] = df['Returns'].rolling(window=20).std() * np.sqrt(252)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    vertical_spacing=0.1,
                    subplot_titles=('Bollinger Bands', 'Annualized Rolling Volatility'),
                    row_width=[0.3, 0.7])

# --- Top Plot: Bollinger Bands ---
fig.add_trace(go.Scatter(x=df['Date'], y=df['Upper_Band'],
                         line=dict(color='rgba(173, 204, 255, 0.2)'),
                         name='Upper Band'), row=1, col=1)

fig.add_trace(go.Scatter(x=df['Date'], y=df['Lower_Band'],
                         line=dict(color='rgba(173, 204, 255, 0.2)'),
                         fill='tonexty', fillcolor='rgba(173, 204, 255, 0.1)',
                         name='Lower Band'), row=1, col=1)

# ADDED: Middle Line (SMA20)
fig.add_trace(go.Scatter(x=df['Date'], y=df['SMA20'],
                         line=dict(color='orange', width=1, dash='dash'),
                         name='Middle Band (SMA20)'), row=1, col=1)

fig.add_trace(go.Scatter(x=df['Date'], y=df['Close'],
                         line=dict(color='green', width=2),
                         name='Close'), row=1, col=1)

# --- Bottom Plot: Volatility ---
fig.add_trace(go.Scatter(x=df['Date'], y=df['Volatility'],
                         line=dict(color='red'), fill='tozeroy',
                         name='Volatility'), row=2, col=1)

fig.update_layout(
    title="Volatility Analysis (Bollinger Bands with Middle SMA)",
    template="plotly_dark",
    showlegend=True, # Enabled legend to distinguish the middle line
    height=800,
    xaxis_rangeslider_visible=False
)

fig.update_yaxes(tickformat=".0%", row=2, col=1)

fig.show()